[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_30_Production_Reliability_Stack.ipynb)

# Lesson 30 — Production Reliability Stack
### Circuit Breakers · Fallback Chains · Canary Deploys · A/B Testing

> **Phase 4 · Track 1 · Lesson 7 of 8** — *Reliability & Safety*
>
> Built on top of: L24 (SLOs + ReliabilityProfiler) · L17 (eval CI gate) · L22 (CostMeter + model routing) · L29 (calibration ECE).

---

## Where we are in the curriculum

Lessons 24 → 29 gave us **measurement**: how to score a single response or a batch (Brier, ECE, ASR, FRR, invariance, faithfulness). Those metrics live in CI, the eval harness, and the nightly scorecard.

But what does your service do **right now**, in production, when the model is misbehaving? You can't wait for the nightly eval to tell you Sonnet's latency p95 just shot to 12 seconds. You need a **live response loop**.

That's L30. We move from offline metrics to **runtime behavior**:

| Pattern | What it does | When it kicks in |
|---|---|---|
| **Circuit Breaker** | Auto-ejects a misbehaving model from rotation | Live signal (errors, latency, calibration) crosses a threshold |
| **Fallback Chain** | Cascades through alternatives | Primary fails or is "open" |
| **Canary Deploy** | Routes 5% to a new model, monitors live | Whenever you ship a new model/prompt |
| **A/B Test** | Statistically rigorous comparison of two variants | Before promoting a candidate to 100% |

By the end, you have a **`ReliabilityRouter`** that wraps last lesson's `CostAwareAgent` and the L24 SLO checks into a single chokepoint your FastAPI handler can call.

---

## Conceptual map — the four patterns

```
                 ┌─────────────────────────────────────┐
                 │           ReliabilityRouter          │
                 │                                      │
   request ─────▶│  1. canary split (5%/95%)            │──▶ response
                 │  2. circuit-breaker check            │
                 │  3. fallback chain (Sonnet→Haiku→…)  │
                 │  4. emit metrics to A/B tracker      │
                 │                                      │
                 └─────────────────────────────────────┘
                                  │
                                  ▼
                         CostMeter (from L22)
                                  │
                                  ▼
                  Prometheus / SQLite / OpenTelemetry
```

Each pattern is a few-dozen lines of Python. The hard part is **picking the thresholds**. That's where L24 SLOs come back.


## 0. Setup

In [ ]:
# Colab / local install. Quiet to keep the notebook tidy.
!pip install anthropic scipy numpy matplotlib pandas -q


In [ ]:
import os
import time
import json
import math
import random
import statistics
from collections import deque, defaultdict
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional, Any
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Anthropic key — Colab Secrets, or environment variable, or paste here.
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."
from anthropic import Anthropic
client = Anthropic()
print("client ready")


---
## 1. Circuit Breakers — the auto-eject

### 1.1 Why

You're routing prod traffic to `claude-sonnet-4-5`. At 14:32 the model's upstream gateway has a partial outage: 30% of calls return 5xx. Your fallback chain catches each error and retries, but every retry costs **another full request**, so your cost graph spikes 4× and latency p95 doubles.

A **circuit breaker** says: *"After N consecutive failures (or N% failure rate in a window), stop trying entirely for T seconds. After T seconds, send ONE probe to check if the upstream recovered. If yes, fully reopen. If no, double T."*

This is the same pattern Netflix's Hystrix popularized — but the LLM version watches **three signals**, not just error rate:

| Signal | Why it matters for LLMs |
|---|---|
| **Error rate** | API 5xx, timeouts, content-policy refusals at the API layer |
| **Latency p95** | Slow responses cascade into queue backlog → user timeouts |
| **Calibration drift (ECE delta)** | The model is "online" but answering wrongly with high confidence |

The third one is the LLM-specific twist. A traditional circuit breaker can't tell a healthy `200 OK` from a `200 OK` that contains a hallucinated answer. With the L29 calibration probe running in the background, you *can*.

### 1.2 The three states

```
        ╔═══════════╗   too many failures   ╔═══════╗
        ║  CLOSED   ║ ─────────────────────▶║ OPEN  ║
        ║ (healthy) ║                       ╚═══════╝
        ╚═══════════╝                           │
              ▲                                 │ cool-down expires
              │ probe success                   ▼
              │                            ╔═══════════╗
              └────────────────────────────║ HALF_OPEN ║
                                           ║  (probe)  ║
                                           ╚═══════════╝
                                                │
                                                │ probe fails
                                                ▼
                                            back to OPEN
```

Let's implement it.


In [ ]:
class BreakerState(str, Enum):
    CLOSED = "closed"        # healthy, traffic flows
    OPEN = "open"            # ejected, traffic blocked
    HALF_OPEN = "half_open"  # one probe in flight to test recovery


@dataclass
class CircuitBreaker:
    name: str
    failure_threshold: int = 5         # consecutive failures to trip
    error_rate_threshold: float = 0.3  # OR 30% failures in window
    latency_p95_threshold_s: float = 8.0
    window_size: int = 20              # rolling window of recent calls
    cooldown_s: float = 30.0           # how long to stay OPEN

    # internal state — don't read directly
    state: BreakerState = BreakerState.CLOSED
    _opened_at: float = 0.0
    _consecutive_failures: int = 0
    _window: deque = field(default_factory=lambda: deque(maxlen=20))

    def __post_init__(self):
        self._window = deque(maxlen=self.window_size)

    # ----- the API your router calls -----
    def allow(self) -> bool:
        '''Return True if a request is allowed through right now.'''
        if self.state == BreakerState.CLOSED:
            return True
        if self.state == BreakerState.OPEN:
            if time.time() - self._opened_at >= self.cooldown_s:
                self.state = BreakerState.HALF_OPEN
                return True  # let exactly one probe through
            return False
        # HALF_OPEN — only the first caller gets through.
        # (For simplicity, single-threaded. In prod, use a lock.)
        return True

    def record_success(self, latency_s: float):
        self._window.append((True, latency_s))
        self._consecutive_failures = 0
        if self.state == BreakerState.HALF_OPEN:
            self.state = BreakerState.CLOSED
            print(f"  [breaker {self.name}] HALF_OPEN → CLOSED (probe succeeded)")

    def record_failure(self, latency_s: float = 0.0):
        self._window.append((False, latency_s))
        self._consecutive_failures += 1
        if self.state == BreakerState.HALF_OPEN:
            self._trip()
            return
        if self._should_trip():
            self._trip()

    # ----- private -----
    def _should_trip(self) -> bool:
        if self._consecutive_failures >= self.failure_threshold:
            return True
        if len(self._window) >= self.window_size:
            errs = sum(1 for ok, _ in self._window if not ok)
            if errs / len(self._window) >= self.error_rate_threshold:
                return True
            lats = sorted(lat for _, lat in self._window if lat > 0)
            if lats:
                p95 = lats[int(0.95 * len(lats))]
                if p95 >= self.latency_p95_threshold_s:
                    return True
        return False

    def _trip(self):
        self.state = BreakerState.OPEN
        self._opened_at = time.time()
        print(f"  [breaker {self.name}] → OPEN (cooldown {self.cooldown_s}s)")


# Tiny demo — simulate a failing service.
random.seed(42)
cb = CircuitBreaker("sonnet", failure_threshold=3, cooldown_s=2.0)

print("--- happy path: 5 successes ---")
for _ in range(5):
    if cb.allow():
        cb.record_success(latency_s=1.2)
print(f"state={cb.state}, consec_fails={cb._consecutive_failures}")

print("\n--- 4 failures in a row (trips at 3) ---")
for i in range(4):
    if cb.allow():
        cb.record_failure(latency_s=0.5)
    else:
        print(f"  request {i} blocked")
print(f"state={cb.state}")

print("\n--- sleep through cooldown, then probe ---")
time.sleep(2.1)
if cb.allow():
    print("  probe allowed (HALF_OPEN)")
    cb.record_success(latency_s=1.1)
print(f"final state={cb.state}")


**💡 EXPERIMENT.** Change `failure_threshold=3` to `1` — does the breaker trip on the *first* failure? Then change the probe to a failure (`record_failure` after the cooldown) — does it go straight back to OPEN with a doubled cooldown? (Hint: our impl doesn't double; that's a one-line change in `_trip`. Add it.)

### 1.3 The calibration twist

Here's the L29 callback. Even when calls succeed, the *model* might be drifting. Plug an ECE probe into the breaker:


In [ ]:
@dataclass
class CalibrationAwareBreaker(CircuitBreaker):
    max_ece: float = 0.15  # from L29 SLO
    _recent_ece: float = 0.0

    def update_ece(self, ece_value: float):
        '''Called by a background job that runs the L29 reliability_diagram probe.'''
        self._recent_ece = ece_value
        if ece_value > self.max_ece and self.state == BreakerState.CLOSED:
            print(f"  [breaker {self.name}] tripping on ECE drift {ece_value:.3f} > {self.max_ece}")
            self._trip()

# Demo: breaker trips even though API calls are succeeding.
cb2 = CalibrationAwareBreaker("sonnet", max_ece=0.10, cooldown_s=2.0)
for _ in range(10):
    if cb2.allow():
        cb2.record_success(latency_s=1.0)
print(f"after 10 happy calls: state={cb2.state}")
cb2.update_ece(0.18)  # ⚠️ calibration drift detected
print(f"after ECE drift:      state={cb2.state}")


---
## 2. Fallback Chains — graceful degradation

### 2.1 Why

When the primary fails, you have four options ranked by quality and cost:

```
  Tier 1: Sonnet 4.5            ← best answer, $$$
  Tier 2: Haiku 4.5             ← decent answer, $
  Tier 3: cached prior answer   ← maybe stale, $0
  Tier 4: static error message  ← honest "we're degraded", $0
```

The principle: **never let a single failure become a user-facing error**. Always serve *something*, and tell the caller *which tier* you served from so they can decorate the UI ("served from cache" badge, "AI is currently degraded" banner).

### 2.2 Implementation

A `FallbackChain` is a list of `(name, callable, breaker)` tuples. We iterate, skipping tiers whose breaker is open, until one returns. Telemetry on every call.


In [ ]:
# Lightweight LRU cache for tier 3.
class TTLCache:
    def __init__(self, max_size: int = 500, ttl_s: float = 3600):
        self.max_size = max_size
        self.ttl_s = ttl_s
        self._store: dict[str, tuple[Any, float]] = {}

    def get(self, key: str):
        if key in self._store:
            value, ts = self._store[key]
            if time.time() - ts < self.ttl_s:
                return value
            del self._store[key]
        return None

    def set(self, key: str, value: Any):
        if len(self._store) >= self.max_size:
            # evict oldest
            oldest = min(self._store, key=lambda k: self._store[k][1])
            del self._store[oldest]
        self._store[key] = (value, time.time())


@dataclass
class FallbackResult:
    answer: str
    tier_name: str       # which tier served
    tier_index: int      # 0 = primary
    latency_s: float
    fallback_used: bool  # True if not the primary

    def __repr__(self):
        return f"<FallbackResult tier={self.tier_index}:{self.tier_name} ({self.latency_s:.2f}s)>"


class FallbackChain:
    def __init__(self, tiers: list[tuple[str, Callable[[str], str], Optional[CircuitBreaker]]]):
        '''Each tier is (name, fn, breaker). breaker can be None for the static-message tail.'''
        self.tiers = tiers
        self.cache = TTLCache()
        self.tier_counts = defaultdict(int)  # telemetry

    def call(self, prompt: str) -> FallbackResult:
        for i, (name, fn, breaker) in enumerate(self.tiers):
            if breaker and not breaker.allow():
                print(f"  [chain] tier {i}:{name} skipped — breaker {breaker.state.value}")
                continue
            t0 = time.time()
            try:
                answer = fn(prompt)
                elapsed = time.time() - t0
                if breaker:
                    breaker.record_success(elapsed)
                self.cache.set(prompt, answer)
                self.tier_counts[name] += 1
                return FallbackResult(
                    answer=answer, tier_name=name, tier_index=i,
                    latency_s=elapsed, fallback_used=(i > 0),
                )
            except Exception as e:
                elapsed = time.time() - t0
                if breaker:
                    breaker.record_failure(elapsed)
                print(f"  [chain] tier {i}:{name} raised {type(e).__name__}: {e}")
        # All tiers exhausted — never reached in practice because the last
        # tier is always the static message that can't fail.
        return FallbackResult(
            answer="(service unavailable)",
            tier_name="exhausted", tier_index=-1,
            latency_s=0.0, fallback_used=True,
        )

    def summary(self) -> pd.DataFrame:
        total = sum(self.tier_counts.values())
        rows = [{"tier": name, "served": count, "share": count / max(total, 1)}
                for name, count in self.tier_counts.items()]
        return pd.DataFrame(rows)


In [ ]:
# Build a real chain: Sonnet → Haiku → cache → static.
# We wrap real Claude calls but inject a 'flaky' flag to demo failures.

class FlakySonnet:
    '''Sonnet wrapper that fails on demand.'''
    def __init__(self, fail_rate: float = 0.0):
        self.fail_rate = fail_rate

    def __call__(self, prompt: str) -> str:
        if random.random() < self.fail_rate:
            raise RuntimeError("upstream 503")
        resp = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text


def haiku(prompt: str) -> str:
    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text


# Tier 3 reads from the shared cache. We need a closure that has access
# to the chain's cache, so we build it inside a factory.
def make_chain(sonnet_fail_rate: float = 0.0):
    sonnet_cb = CircuitBreaker("sonnet", failure_threshold=2, cooldown_s=5.0)
    haiku_cb  = CircuitBreaker("haiku",  failure_threshold=3, cooldown_s=5.0)
    chain_ref = {}  # capture chain after creation for the cache closure

    def from_cache(prompt: str) -> str:
        cached = chain_ref["chain"].cache.get(prompt)
        if cached is None:
            raise RuntimeError("cache miss")
        return f"[cached] {cached}"

    def static_message(prompt: str) -> str:
        return "I'm temporarily degraded. Please try again in a moment."

    chain = FallbackChain([
        ("sonnet",  FlakySonnet(fail_rate=sonnet_fail_rate), sonnet_cb),
        ("haiku",   haiku,           haiku_cb),
        ("cache",   from_cache,      None),
        ("static",  static_message,  None),
    ])
    chain_ref["chain"] = chain
    return chain


# ----- Demo: Sonnet healthy -----
print("=== Sonnet healthy (fail_rate=0) ===")
chain = make_chain(sonnet_fail_rate=0.0)
r = chain.call("In one sentence, what's a vector database?")
print(r)
print(r.answer[:120], "...")

# ----- Demo: Sonnet flaky -----
print("\n=== Sonnet 100% failing — chain falls through to Haiku ===")
chain2 = make_chain(sonnet_fail_rate=1.0)
r2 = chain2.call("In one sentence, what's a vector database?")
print(r2)
print(r2.answer[:120], "...")


**💡 EXPERIMENT.** Run the same prompt twice through `chain2`. Watch what happens after the Sonnet breaker trips and a prior `r2.answer` is in the cache. Which tier serves the second call? Why is "cache" higher than "haiku" in the chain order — what's the cost/quality tradeoff if you swap them?


---
## 3. Canary Deploys — ship without breaking prod

### 3.1 Why

You've trained/picked a new candidate model — maybe Sonnet 4.6 just launched, or you've tuned a prompt, or you fine-tuned a Haiku for your domain. You want to ship it, but you can't afford to flip 100% of traffic and discover the FRR doubled.

The canary pattern: **route a small percentage of traffic to the candidate, monitor live metrics, auto-rollback if any guardrail metric regresses, gradually increase if all clear.**

```
   t = 0:   5% candidate / 95% baseline   ← watch ECE, FRR, ASR, p95, $/req
   t = 1h:  if green → 25%
   t = 4h:  if green → 50%
   t = 24h: if green → 100%
   any time: if red → 0% (instant rollback)
```

The L24 SLOs are your guardrails. The L17 eval gate is your *offline* check. The canary is the *online* check.

### 3.2 Implementation

Three pieces: a deterministic split (so the same user sees consistent variant on retries), a live metric buffer per variant, and a comparator that triggers rollback.


In [ ]:
@dataclass
class CanaryConfig:
    candidate_share: float = 0.05      # 5%
    min_samples_for_compare: int = 50  # don't compare with 3 calls of noise
    max_ece_regression: float = 0.05   # candidate ECE can be up to +0.05 vs baseline
    max_frr_regression: float = 0.05
    max_p95_regression_s: float = 2.0
    sticky_by_user: bool = True


def hash_to_unit(user_id: str) -> float:
    '''Stable 0..1 mapping. Same user → same bucket → same variant.'''
    import hashlib
    h = hashlib.sha256(user_id.encode()).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


@dataclass
class VariantStats:
    name: str
    n: int = 0
    errors: int = 0
    latencies: list = field(default_factory=list)
    refusals: int = 0
    ece_values: list = field(default_factory=list)  # appended by background ECE probe

    @property
    def error_rate(self): return self.errors / max(self.n, 1)
    @property
    def refusal_rate(self): return self.refusals / max(self.n, 1)
    @property
    def p95(self):
        if not self.latencies: return 0.0
        s = sorted(self.latencies); return s[int(0.95 * len(s))]
    @property
    def mean_ece(self):
        return statistics.mean(self.ece_values) if self.ece_values else 0.0


class CanaryRouter:
    def __init__(self, baseline_fn: Callable, candidate_fn: Callable, config: CanaryConfig):
        self.baseline_fn = baseline_fn
        self.candidate_fn = candidate_fn
        self.config = config
        self.baseline_stats = VariantStats("baseline")
        self.candidate_stats = VariantStats("candidate")
        self.rolled_back = False

    def route(self, prompt: str, user_id: str) -> tuple[str, str]:
        '''Returns (answer, variant_name). variant_name lets caller emit metrics.'''
        if self.rolled_back:
            variant = "baseline"
        else:
            bucket = hash_to_unit(user_id) if self.config.sticky_by_user else random.random()
            variant = "candidate" if bucket < self.config.candidate_share else "baseline"

        fn = self.candidate_fn if variant == "candidate" else self.baseline_fn
        stats = self.candidate_stats if variant == "candidate" else self.baseline_stats

        t0 = time.time()
        try:
            answer = fn(prompt)
            elapsed = time.time() - t0
            stats.n += 1
            stats.latencies.append(elapsed)
            if "I can't help" in answer or "I cannot" in answer:
                stats.refusals += 1
        except Exception as e:
            elapsed = time.time() - t0
            stats.n += 1; stats.errors += 1
            stats.latencies.append(elapsed)
            answer = "(error)"

        self._maybe_rollback()
        return answer, variant

    def _maybe_rollback(self):
        if self.rolled_back: return
        b, c = self.baseline_stats, self.candidate_stats
        if c.n < self.config.min_samples_for_compare: return

        reasons = []
        if c.error_rate - b.error_rate > 0.05:
            reasons.append(f"error_rate +{(c.error_rate - b.error_rate):.1%}")
        if c.refusal_rate - b.refusal_rate > self.config.max_frr_regression:
            reasons.append(f"FRR +{(c.refusal_rate - b.refusal_rate):.1%}")
        if c.p95 - b.p95 > self.config.max_p95_regression_s:
            reasons.append(f"p95 +{c.p95 - b.p95:.1f}s")
        if c.mean_ece and b.mean_ece and c.mean_ece - b.mean_ece > self.config.max_ece_regression:
            reasons.append(f"ECE +{c.mean_ece - b.mean_ece:.3f}")

        if reasons:
            self.rolled_back = True
            print(f"  [canary] ROLLBACK: {', '.join(reasons)}")

    def report(self) -> pd.DataFrame:
        rows = []
        for s in [self.baseline_stats, self.candidate_stats]:
            rows.append({
                "variant": s.name,
                "n": s.n,
                "error_rate": s.error_rate,
                "refusal_rate": s.refusal_rate,
                "p95_s": s.p95,
                "mean_ece": s.mean_ece,
            })
        return pd.DataFrame(rows)


In [ ]:
# Simulate a canary deploy where the candidate is secretly worse on FRR.
def fake_baseline(prompt: str) -> str:
    time.sleep(random.uniform(0.05, 0.15))
    if random.random() < 0.05:  # 5% benign refusals
        return "I can't help with that."
    return "Sure: a vector database stores embeddings."

def fake_candidate(prompt: str) -> str:
    time.sleep(random.uniform(0.05, 0.20))
    if random.random() < 0.25:  # ⚠️ 25% refusals — paranoid candidate
        return "I cannot help with that request."
    return "Sure: a vector database stores embeddings."

router = CanaryRouter(fake_baseline, fake_candidate, CanaryConfig(
    candidate_share=0.20,           # 20% to make demo fast
    min_samples_for_compare=30,
))

random.seed(7)
for i in range(300):
    router.route("what is a vector database?", user_id=f"user_{i}")
    if router.rolled_back:
        print(f"  rolled back after {i+1} requests")
        break

print(router.report())


**💡 EXPERIMENT.** Try `candidate_share=0.05`. Notice the rollback happens later — fewer candidate samples means it takes longer to accumulate `min_samples_for_compare`. This is the bias/variance dial: smaller canary = safer on the downside, slower to detect a bad candidate.


---
## 4. A/B Testing — when "candidate looks better" is actually noise

### 4.1 Why this matters

After 24 hours your canary has 800 samples; baseline has 12,000. You eyeball:

| | baseline | candidate |
|---|---|---|
| refusal_rate | 4.2% | 3.1% |

"Candidate is better, ship it." But is it? Maybe with 800 samples and 31 refusals, the difference is just noise. The right question is: **what's the probability you'd see a 1.1-point gap by chance if the true rates were identical?**

That's a hypothesis test. Two tests cover most LLM A/Bs:

- **Two-proportion z-test** — for binary outcomes: refused/answered, hallucinated/grounded, jailbroken/blocked. Refusal rate, ASR, FRR, faithfulness pass-rate all live here.
- **Welch's t-test** — for continuous outcomes with possibly different variances: latency, cost per request, token count, judge score.

We'll implement both from `scipy`, then talk about the trap.

### 4.2 Sample size math — *before* you ship

The cardinal sin: peek at results, see significance, declare victory. The honest move is to compute **how many samples you need** *before* the test, based on the effect size you care about.

For two proportions, the required sample per arm to detect an absolute difference `δ` with 80% power at α=0.05 is approximately:

```
n ≈ 2 · p̄(1-p̄) · (z_{α/2} + z_β)² / δ²
```

where `p̄` is the pooled baseline rate.


In [ ]:
def sample_size_two_proportions(p_baseline: float, mde: float,
                                  alpha: float = 0.05, power: float = 0.80) -> int:
    '''
    Required n PER ARM to detect absolute difference `mde` with given power.
    p_baseline: expected baseline rate (e.g. 0.04 for 4% FRR)
    mde:        minimum detectable effect (e.g. 0.01 for 1pp absolute)
    '''
    z_a = stats.norm.ppf(1 - alpha / 2)
    z_b = stats.norm.ppf(power)
    p_bar = p_baseline + mde / 2  # midpoint
    var = 2 * p_bar * (1 - p_bar)
    n = var * (z_a + z_b) ** 2 / mde ** 2
    return int(math.ceil(n))


for p, mde in [(0.04, 0.01), (0.04, 0.005), (0.5, 0.02), (0.1, 0.02)]:
    n = sample_size_two_proportions(p, mde)
    print(f"baseline={p:.2%}, MDE={mde:.2%} → need n={n:,} per arm")


Notice: detecting a 0.5pp swing in a 4% FRR needs ~12k requests per arm. That's why **statistical rigor punishes premature decisions**. If you want to ship faster, either accept a bigger MDE, lower your power, or run a bigger canary share.

### 4.3 The tests


In [ ]:
@dataclass
class ABResult:
    metric: str
    baseline_summary: str
    candidate_summary: str
    delta: float
    p_value: float
    significant: bool
    direction: str  # "candidate_better" | "candidate_worse" | "no_difference"

    def __repr__(self):
        sig = "✅ SIG" if self.significant else "❌ ns"
        return f"<AB {self.metric}: Δ={self.delta:+.4f}, p={self.p_value:.4f} [{sig}] {self.direction}>"


def two_proportion_test(name: str,
                        baseline_success: int, baseline_n: int,
                        candidate_success: int, candidate_n: int,
                        alpha: float = 0.05,
                        lower_is_better: bool = True) -> ABResult:
    '''
    Two-sided z-test for proportions. Reports candidate vs baseline.
    For metrics like refusal_rate or ASR, lower_is_better=True.
    '''
    p_b = baseline_success / baseline_n
    p_c = candidate_success / candidate_n
    p_pool = (baseline_success + candidate_success) / (baseline_n + candidate_n)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / baseline_n + 1 / candidate_n))
    z = (p_c - p_b) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    delta = p_c - p_b
    sig = p_value < alpha
    if not sig:
        direction = "no_difference"
    elif (delta < 0 and lower_is_better) or (delta > 0 and not lower_is_better):
        direction = "candidate_better"
    else:
        direction = "candidate_worse"
    return ABResult(
        metric=name,
        baseline_summary=f"{p_b:.2%} ({baseline_success}/{baseline_n})",
        candidate_summary=f"{p_c:.2%} ({candidate_success}/{candidate_n})",
        delta=delta, p_value=p_value, significant=sig, direction=direction,
    )


def welch_t_test(name: str,
                 baseline_values: list[float], candidate_values: list[float],
                 alpha: float = 0.05,
                 lower_is_better: bool = True) -> ABResult:
    '''Welch's t-test for means (unequal variance). For latency, cost, scores.'''
    b = np.array(baseline_values); c = np.array(candidate_values)
    t_stat, p_value = stats.ttest_ind(c, b, equal_var=False)
    delta = c.mean() - b.mean()
    sig = p_value < alpha
    if not sig:
        direction = "no_difference"
    elif (delta < 0 and lower_is_better) or (delta > 0 and not lower_is_better):
        direction = "candidate_better"
    else:
        direction = "candidate_worse"
    return ABResult(
        metric=name,
        baseline_summary=f"mean={b.mean():.3f} (n={len(b)})",
        candidate_summary=f"mean={c.mean():.3f} (n={len(c)})",
        delta=delta, p_value=p_value, significant=sig, direction=direction,
    )


# ----- Demo on the canary data we just collected -----
b_stats = router.baseline_stats
c_stats = router.candidate_stats

r1 = two_proportion_test(
    "refusal_rate",
    baseline_success=b_stats.refusals, baseline_n=b_stats.n,
    candidate_success=c_stats.refusals, candidate_n=c_stats.n,
    lower_is_better=True,
)
print(r1)

r2 = welch_t_test(
    "latency_s",
    baseline_values=b_stats.latencies,
    candidate_values=c_stats.latencies,
    lower_is_better=True,
)
print(r2)


### 4.4 The sequential-testing trap

You ran the canary, peeked at p-values every 5 minutes, saw p<0.05 at hour 3, and rolled back. **You probably over-rolled-back.** Peeking inflates your false-positive rate: under repeated independent looks, the chance of *some* look hitting p<0.05 grows well past 5%.

Two honest fixes:

1. **Pre-commit to a sample size.** Compute `n` once with `sample_size_two_proportions`, then run the test exactly once at that n. No peeking.
2. **Use a sequential test.** Adjusted procedures (Pocock, O'Brien-Fleming, mSPRT, Bayesian posterior) explicitly allow peeking. Production A/B platforms (LaunchDarkly, Statsig, GrowthBook) ship these. Roll your own only if you must.

For canary rollback, the *honest* approach is:

- **Big regressions** (e.g. ECE doubled, latency p95 +5s) → trigger immediate rollback with no significance test. The effect is large enough that you trust your eyes.
- **Small regressions** (e.g. FRR +0.5pp) → wait for the pre-committed sample size before deciding.

The rollback rule in our `CanaryRouter._maybe_rollback` uses thresholded deltas — not p-values — for exactly this reason. It's the "big regression" gate. The statistical tests are for the **promotion** decision after the canary period ends.

### 4.5 Putting the test in CI


In [ ]:
# Example CI gate: after a 24h canary window, decide promote vs rollback.
def promotion_decision(canary_report: dict) -> dict:
    '''Returns {decision: 'promote'|'hold'|'rollback', reasons: [...]}.'''
    reasons = []
    decision = "promote"

    # Run the binary tests. lower_is_better=True for refusal/error/ASR.
    for metric in ["refusal_rate", "error_rate"]:
        r = two_proportion_test(
            metric,
            canary_report[metric]["baseline_success"],
            canary_report[metric]["baseline_n"],
            canary_report[metric]["candidate_success"],
            canary_report[metric]["candidate_n"],
            lower_is_better=True,
        )
        reasons.append(str(r))
        if r.direction == "candidate_worse":
            decision = "rollback"
        elif r.direction == "no_difference" and decision == "promote":
            decision = "hold"  # not worse, but also not better — don't auto-promote

    return {"decision": decision, "reasons": reasons}


fake_report = {
    "refusal_rate": dict(baseline_success=420, baseline_n=10000,
                         candidate_success=22, candidate_n=500),
    "error_rate":  dict(baseline_success=80, baseline_n=10000,
                        candidate_success=4, candidate_n=500),
}
result = promotion_decision(fake_report)
print("decision:", result["decision"])
for r in result["reasons"]:
    print(" -", r)


---
## 5. Capstone — `ReliabilityRouter`

Now we wire all four patterns into a single class that:

1. **Takes a request** with optional `user_id`.
2. **Splits canary**: 5% to candidate model, 95% to baseline.
3. **Inside each variant**, runs a **fallback chain** (Sonnet → Haiku → cache → static).
4. **Each tier guarded by a circuit breaker** watching error rate + latency + ECE.
5. **Emits metrics** the A/B test harness reads at the end of the window.
6. **Wraps** the L22 `CostMeter` so every call counts $.

This is the single chokepoint your FastAPI endpoint calls. It's ~100 lines because every piece is composable.


In [ ]:
# Lightweight CostMeter from L22 (just enough to demo wrapping).
PRICING = {
    "claude-sonnet-4-5":   dict(input_per_mtok=3.00, output_per_mtok=15.00),
    "claude-haiku-4-5":    dict(input_per_mtok=1.00, output_per_mtok=5.00),
}

class CostMeter:
    def __init__(self):
        self.rows: list[dict] = []
    def record(self, model: str, in_toks: int, out_toks: int, tag: str = ""):
        p = PRICING.get(model, dict(input_per_mtok=0, output_per_mtok=0))
        cost = in_toks * p["input_per_mtok"] / 1e6 + out_toks * p["output_per_mtok"] / 1e6
        self.rows.append(dict(model=model, in_toks=in_toks, out_toks=out_toks,
                              cost=cost, tag=tag, ts=time.time()))
        return cost
    def total(self): return sum(r["cost"] for r in self.rows)
    def by_tag(self):
        df = pd.DataFrame(self.rows)
        return df.groupby("tag")["cost"].sum() if not df.empty else pd.Series()


def claude_call(model: str, prompt: str, meter: CostMeter, tag: str) -> str:
    resp = client.messages.create(
        model=model, max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    meter.record(model, resp.usage.input_tokens, resp.usage.output_tokens, tag=tag)
    return resp.content[0].text


In [ ]:
class ReliabilityRouter:
    '''
    The single chokepoint. Wraps:
      - canary (% to candidate)
      - circuit breakers per model
      - fallback chain per variant
      - cost meter
      - A/B metric buckets
    '''
    def __init__(self,
                 baseline_model: str = "claude-sonnet-4-5",
                 candidate_model: str = "claude-haiku-4-5",
                 cost_meter: Optional[CostMeter] = None,
                 canary_config: Optional[CanaryConfig] = None):
        self.baseline_model = baseline_model
        self.candidate_model = candidate_model
        self.meter = cost_meter or CostMeter()
        self.canary_config = canary_config or CanaryConfig()

        self.baseline_cb = CircuitBreaker(baseline_model, failure_threshold=3, cooldown_s=10)
        self.candidate_cb = CircuitBreaker(candidate_model, failure_threshold=3, cooldown_s=10)
        self.haiku_fallback_cb = CircuitBreaker("haiku-fallback", failure_threshold=3, cooldown_s=10)

        self.cache = TTLCache()
        self.baseline_stats = VariantStats("baseline")
        self.candidate_stats = VariantStats("candidate")
        self.rolled_back = False
        self.tier_counts = defaultdict(int)

    def _build_chain(self, primary_model: str, primary_cb: CircuitBreaker, tag: str) -> FallbackChain:
        def primary(prompt: str) -> str:
            return claude_call(primary_model, prompt, self.meter, tag=f"{tag}:primary")

        def secondary(prompt: str) -> str:
            return claude_call("claude-haiku-4-5", prompt, self.meter, tag=f"{tag}:fallback")

        def from_cache(prompt: str) -> str:
            cached = self.cache.get(prompt)
            if cached is None: raise RuntimeError("cache miss")
            return f"[cached] {cached}"

        def static_msg(prompt: str) -> str:
            return "We're temporarily degraded. Please try again shortly."

        return FallbackChain([
            (primary_model, primary,   primary_cb),
            ("haiku-fb",    secondary, self.haiku_fallback_cb),
            ("cache",       from_cache, None),
            ("static",      static_msg, None),
        ])

    def call(self, prompt: str, user_id: str = "anon") -> dict:
        # 1. canary split
        if self.rolled_back:
            variant = "baseline"
        else:
            bucket = hash_to_unit(user_id) if self.canary_config.sticky_by_user else random.random()
            variant = "candidate" if bucket < self.canary_config.candidate_share else "baseline"

        if variant == "candidate":
            chain = self._build_chain(self.candidate_model, self.candidate_cb, tag="candidate")
            stats = self.candidate_stats
        else:
            chain = self._build_chain(self.baseline_model, self.baseline_cb, tag="baseline")
            stats = self.baseline_stats

        # 2. run the chain
        result = chain.call(prompt)

        # 3. cache primary success
        if result.tier_index == 0:
            self.cache.set(prompt, result.answer)

        # 4. metrics
        stats.n += 1
        stats.latencies.append(result.latency_s)
        if result.tier_name == "static" or "I can't help" in result.answer or "I cannot" in result.answer:
            stats.refusals += 1
        if result.tier_index == -1:
            stats.errors += 1
        self.tier_counts[result.tier_name] += 1

        # 5. rollback check (uses thresholds, not p-values)
        self._maybe_rollback()

        return {
            "answer": result.answer,
            "variant": variant,
            "tier": result.tier_name,
            "latency_s": result.latency_s,
            "fallback_used": result.fallback_used,
            "cost_so_far": self.meter.total(),
        }

    def _maybe_rollback(self):
        if self.rolled_back: return
        b, c = self.baseline_stats, self.candidate_stats
        if c.n < self.canary_config.min_samples_for_compare: return
        reasons = []
        if c.error_rate - b.error_rate > 0.05:
            reasons.append(f"error +{(c.error_rate - b.error_rate):.1%}")
        if c.refusal_rate - b.refusal_rate > self.canary_config.max_frr_regression:
            reasons.append(f"FRR +{(c.refusal_rate - b.refusal_rate):.1%}")
        if c.p95 - b.p95 > self.canary_config.max_p95_regression_s:
            reasons.append(f"p95 +{c.p95 - b.p95:.1f}s")
        if reasons:
            self.rolled_back = True
            print(f"  [router] CANARY ROLLBACK: {', '.join(reasons)}")

    def scorecard(self) -> pd.DataFrame:
        rows = []
        for s in [self.baseline_stats, self.candidate_stats]:
            rows.append(dict(variant=s.name, n=s.n,
                             error_rate=s.error_rate, refusal_rate=s.refusal_rate,
                             p95_s=s.p95))
        return pd.DataFrame(rows)

    def tier_breakdown(self) -> pd.DataFrame:
        return pd.DataFrame([{"tier": t, "served": n} for t, n in self.tier_counts.items()])

    def ab_test(self) -> list[ABResult]:
        b, c = self.baseline_stats, self.candidate_stats
        results = []
        results.append(two_proportion_test(
            "refusal_rate", b.refusals, b.n, c.refusals, c.n, lower_is_better=True))
        results.append(two_proportion_test(
            "error_rate", b.errors, b.n, c.errors, c.n, lower_is_better=True))
        results.append(welch_t_test(
            "latency_s", b.latencies, c.latencies, lower_is_better=True))
        return results


In [ ]:
# ----- Live demo (smaller traffic to keep API spend low) -----
router = ReliabilityRouter(
    baseline_model="claude-sonnet-4-5",
    candidate_model="claude-haiku-4-5",
    canary_config=CanaryConfig(candidate_share=0.30, min_samples_for_compare=10),
)

prompts = [
    "In one sentence, what is a vector database?",
    "In one sentence, what is RAG?",
    "In one sentence, what is fine-tuning?",
    "In one sentence, what is a transformer?",
]

random.seed(99)
for i in range(20):
    p = random.choice(prompts)
    out = router.call(p, user_id=f"user_{i}")
    print(f"  call {i:02d}: variant={out['variant']:9s} tier={out['tier']:8s} "
          f"latency={out['latency_s']:.2f}s")

print("\n=== scorecard ===")
print(router.scorecard())
print("\n=== tier breakdown ===")
print(router.tier_breakdown())
print(f"\ntotal cost: ${router.meter.total():.4f}")
print("\n=== A/B test ===")
for r in router.ab_test():
    print(" ", r)


**💡 EXPERIMENT.** With only 20 samples and ~6 in the candidate arm, *every* A/B test reports `no_difference`. That's correct behavior — not a bug. Re-run with 200 calls and watch latency become significant (Haiku is faster than Sonnet). Confidence comes from sample size, not from the test name being scary.

---

## 6. Where this lives in AutoResearcher

L31's capstone will package these patterns into the open-source repo as:

```
auto_researcher/
  reliability/
    breakers.py          ← CircuitBreaker, CalibrationAwareBreaker
    fallback.py          ← FallbackChain, TTLCache
    canary.py            ← CanaryRouter, CanaryConfig, hash_to_unit
    ab.py                ← two_proportion_test, welch_t_test, sample_size_…
    router.py            ← ReliabilityRouter (single entry point)
  evals/
    canary_promotion.py  ← CI script that reads 24h window + emits decision
.github/workflows/
  canary-promote.yml     ← runs daily, promotes/rolls-back automatically
```

Your FastAPI handler from L16 changes from:

```python
@app.post("/research")
def research(req: ResearchRequest):
    return cost_aware_agent.run(req.query)
```

to:

```python
@app.post("/research")
def research(req: ResearchRequest, user: User = Depends(...)):
    return reliability_router.call(req.query, user_id=user.id)
```

That's it. One line change at the call site; all the resilience moves inside the router.

---

## 7. Pitfalls (the field-guide list)

1. **Breaker thresholds copied from a blog post.** Your traffic ≠ Netflix's. Start permissive, watch the rolling p95 in prod for a week, then tighten.
2. **Cooldown too short → flapping.** Breaker trips, half-opens at 5s, fails again, opens, ad infinitum. Exponential backoff is the fix.
3. **Cache poisoning.** Tier 3 serves a stale wrong answer forever because TTL is too long. Cap TTL at minutes for high-stakes domains.
4. **Sticky canary on the wrong key.** Bucketing by request ID instead of user ID means a single user oscillates between variants — bad UX and breaks your stats (within-user variance pollutes between-arm comparison). Always hash a *stable* identifier.
5. **Peeking at p-values.** Covered above. Pre-commit your n, or use sequential tests.
6. **Confusing "no significant difference" with "they're the same".** Absence of evidence ≠ evidence of absence. Report confidence intervals, not just p-values.
7. **Rollback rule that never triggers.** A canary with thresholds set so loose that even a 3× ECE regression doesn't trip it. Test the rollback path in staging by deploying a deliberately broken candidate.
8. **Cost blowup from fallbacks.** Every fallback is a *second* API call. A 5% Sonnet failure rate means 5% of traffic costs Sonnet $ + Haiku $. Plan capacity accordingly.
9. **Breaker observability black hole.** Trips silently, you find out the next morning. Wire breaker state-change events to Slack/Pager/Sentry. Always.
10. **Treating canary as a stopping rule.** Canary is only the *first* safety net. The L17 eval gate is the second. The L24 SLO scorecard is the third. Stack them; don't replace one with the other.

---

## 8. Recap

You can now:

- Implement a 3-state circuit breaker watching error rate, latency p95, and ECE drift
- Compose a fallback chain that gracefully degrades through model tiers, cache, and static text
- Run a canary deploy with sticky user bucketing and threshold-based auto-rollback
- Run a proper two-proportion z-test and Welch's t-test for promote/hold/rollback decisions
- Compute pre-commit sample sizes so you don't ship on noise
- Compose all four into a single `ReliabilityRouter` your service can call

**Next (L31):** the **Track 1 Capstone** — port `breakers.py`, `fallback.py`, `canary.py`, `ab.py`, and `router.py` into the AutoResearcher repo with full tests + a CI workflow that promotes/rolls-back canaries on a schedule. That's the open-source portfolio piece showing you can run an LLM service that doesn't fall over.

See you tomorrow. 🚀
